# 02 - Prepare Training Data

**RAPP Lab 04 - Vision-Action Model**

This notebook prepares the processed CSV data for training the Action Chunking Transformer:

1. **Discover Episodes:** Load metadata for all processed recordings
2. **Split by Recording:** Assign entire episodes to train/val/test (70/15/15)
3. **Compute Normalization:** Standardize features using training stats only (no data leakage)
4. **Generate Windows:** Episode-aware sliding windows (never cross episode boundaries)
5. **Create PyTorch Datasets:** With on-the-fly augmentation for training
6. **Visualize & Validate:** Inspect distributions, sample windows, and batch shapes
7. **Save to Disk:** Preprocessed tensors, normalization stats, and split assignments

---

**Input:** CSV files from `01_process_rosbags.ipynb` 
**Output:** Preprocessed tensors in `/data/processed/tensors/`

## 1. Imports and Configuration

In [ ]:
import sys
from pathlib import Path

# Ensure vam_utils is importable
workspace_dir = Path('/workspace')
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))

import json
import logging
from dataclasses import asdict

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from vam_utils.config.data_config import DataPipelineConfig
from vam_utils.data import (
    EpisodeLoader,
    split_episodes_by_recording,
    compute_normalization_stats,
    apply_normalization,
    compute_window_indices,
    VAMDataset,
    create_dataloaders,
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')
logger = logging.getLogger('prepare_data')

print('Imports successful!')

In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION — Change these to tune window sizes
# =============================================================================
from datetime import datetime

DATE = datetime.now().strftime("%Y_%m_%d")

config = DataPipelineConfig(
    experiment_name=f"{DATE}_tin10_tout10", # <-- Name this experiment (creates output subdir)
    T_in=10,                                # <-- Input context window (frames)
    T_out=10,                               # <-- Prediction horizon (frames)
    stride=1,                               # <-- Sliding window stride
    batch_size=32,                          # <-- Training batch size
    temporal_jitter_max=2,                  # <-- Augmentation: +/- frames jitter
    skeleton_noise_std=0.01,                # <-- Augmentation: skeleton noise (meters)
)

# =============================================================================
# At ~15Hz:  T_in=10 -> 0.67s context,  T_out=10 -> 0.67s prediction
# To experiment: change T_out to 15 or 20, rename experiment, rerun notebook.
#
# To reload a previous experiment's config:
#   config = DataPipelineConfig.load("/data/processed/tensors/2026_02_10_tin10_tout15/pipeline_config.json")
# =============================================================================

print('Data Pipeline Configuration')
print('=' * 50)
print(f'Experiment:            {config.experiment_name}')
print(f'Input window (T_in):   {config.T_in} frames  (~{config.T_in / 15:.2f}s at 15Hz)')
print(f'Output window (T_out): {config.T_out} frames  (~{config.T_out / 15:.2f}s at 15Hz)')
print(f'Min episode length:    {config.min_episode_length} frames (auto)')
print(f'Stride:                {config.stride}')
print(f'Input dim:             {config.input_dim} (skeleton={config.skeleton_dim} + joints={config.joint_dim})')
print(f'Split ratios:          {config.train_ratio}/{config.val_ratio}/{config.test_ratio}')
print(f'Batch size:            {config.batch_size}')
print(f'Augmentation:          {"ON" if config.augmentation_enabled else "OFF"}')
if config.augmentation_enabled:
    print(f'  Temporal jitter:     +/- {config.temporal_jitter_max} frames (margin={config.jitter_margin} auto)')
    print(f'  Skeleton noise:      sigma={config.skeleton_noise_std}m')
print(f'\nCSV directory:         {config.csv_dir}')
print(f'Output directory:       {config.output_dir}')

## 2. Discover and Validate Episodes

In [ ]:
loader = EpisodeLoader(config)
episodes = loader.discover_episodes()

print(f'Discovered {len(episodes)} episodes\n')
print(f'{"Episode ID":<35} {"Frames":>7} {"Duration":>10} {"Freq (Hz)":>10} {"NaN":>5} {"Jumps":>6} {"JointViol":>10}')
print('-' * 95)

total_frames = 0
total_duration = 0.0
for ep in episodes:
    print(
        f'{ep.episode_id:<35} {ep.num_frames:>7} '
        f'{ep.duration_sec:>9.1f}s {ep.avg_frequency_hz:>10.2f} '
        f'{ep.quality_nan_count:>5} {ep.quality_jumps:>6} {ep.quality_joint_violations:>10}'
    )
    total_frames += ep.num_frames
    total_duration += ep.duration_sec

print('-' * 95)
print(f'{"TOTAL":<35} {total_frames:>7} {total_duration:>9.1f}s')

# Flag short episodes
short = [ep for ep in episodes if ep.num_frames < config.min_episode_length]
if short:
    print(f'\nWARNING: {len(short)} episode(s) are shorter than {config.min_episode_length} frames and will produce 0 windows')
    for ep in short:
        print(f'  {ep.episode_id}: {ep.num_frames} frames')

In [ ]:
# Validate each episode
print('Validating episodes...\n')

episode_data_raw = {}
for ep in episodes:
    arrays = loader.load_episode_arrays(ep)
    warnings = loader.validate_episode(ep, arrays)
    episode_data_raw[ep.episode_id] = arrays

    status = 'PASS' if not warnings else 'WARN'
    print(f'  {ep.episode_id}: {status}')
    for w in warnings:
        print(f'    - {w}')

print(f'\nLoaded {len(episode_data_raw)} episodes into memory')

## 3. Split Episodes by Recording

In [ ]:
splits = split_episodes_by_recording(
    episodes,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    seed=config.split_seed,
)

print('Episode Split Assignment')
print('=' * 70)
for split_name in ['train', 'val', 'test']:
    eps = splits[split_name]
    total = sum(e.num_frames for e in eps)
    print(f'\n{split_name.upper()} ({len(eps)} episodes, {total} frames):')
    for ep in eps:
        print(f'  {ep.episode_id}  ({ep.num_frames} frames, {ep.duration_sec:.1f}s)')

## 4. Compute Normalization Statistics

Statistics are computed from **training episodes only** to prevent data leakage.

**No hip-centering** — skeleton data stays in `robot_base_link` frame. The performer's absolute
position relative to the robot is meaningful (grid positions at different radii and arc locations).
We only apply standardization (zero mean, unit variance).

In [ ]:
# Collect training episode data for normalization
train_episode_data = [episode_data_raw[ep.episode_id] for ep in splits['train']]

print(f'Computing normalization stats from {len(train_episode_data)} training episodes...\n')
norm_stats = compute_normalization_stats(train_episode_data)

print('Skeleton Statistics (48 features, robot_base_link frame):')
print(f'  Mean range: [{norm_stats.skeleton_mean.min():.4f}, {norm_stats.skeleton_mean.max():.4f}]')
print(f'  Std range:  [{norm_stats.skeleton_std.min():.4f}, {norm_stats.skeleton_std.max():.4f}]')

print(f'\nJoint Statistics (6 features, radians):')
print(f'  Mean: {np.array2string(norm_stats.joint_mean, precision=4, separator=", ")}')
print(f'  Std:  {np.array2string(norm_stats.joint_std, precision=4, separator=", ")}')

print(f'\nEnd-Effector Statistics (6 features, meters + radians):')
print(f'  Mean: {np.array2string(norm_stats.ee_mean, precision=4, separator=", ")}')
print(f'  Std:  {np.array2string(norm_stats.ee_std, precision=4, separator=", ")}')

### 4.1 Visualize Feature Distributions

In [ ]:
# Visualize skeleton and joint distributions before/after standardization

# Gather all training frames
all_train_skeleton = np.concatenate([episode_data_raw[ep.episode_id]['skeleton'] for ep in splits['train']])
all_train_joints = np.concatenate([episode_data_raw[ep.episode_id]['joints'] for ep in splits['train']])

# Standardize
all_train_skeleton_normed, all_train_joints_normed = apply_normalization(
    all_train_skeleton, all_train_joints, norm_stats
)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Skeleton Coords (raw, meters)',
        'Skeleton Coords (standardized)',
        'Joint Angles (raw, radians)',
        'Joint Angles (standardized)',
    )
)

# Skeleton raw - sample 3 representative features
sk_labels = ['sk_0_x (pelvis)', 'sk_7_y (L hand)', 'sk_11_z (R hand)']
sk_indices = [0, 22, 35]  # sk_0_x, sk_7_y, sk_11_z
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, (idx, label) in enumerate(zip(sk_indices, sk_labels)):
    fig.add_trace(go.Histogram(x=all_train_skeleton[:, idx], name=label, marker_color=colors[i],
                               opacity=0.7, nbinsx=50, showlegend=True), row=1, col=1)
    fig.add_trace(go.Histogram(x=all_train_skeleton_normed[:, idx], name=label + ' (std)', marker_color=colors[i],
                               opacity=0.7, nbinsx=50, showlegend=False), row=1, col=2)

# Joints raw
joint_colors = ['#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
for j in range(6):
    fig.add_trace(go.Histogram(x=all_train_joints[:, j], name=f'j{j}', marker_color=joint_colors[j],
                               opacity=0.7, nbinsx=50, showlegend=True), row=2, col=1)
    fig.add_trace(go.Histogram(x=all_train_joints_normed[:, j], name=f'j{j} (std)', marker_color=joint_colors[j],
                               opacity=0.7, nbinsx=50, showlegend=False), row=2, col=2)

fig.update_layout(height=700, width=1200, title_text='Feature Distributions (Training Data)',
                  barmode='overlay')
fig.show()

print('Standardized features should be approximately centered at 0 with unit variance.')

## 5. Generate Window Indices

Windows are generated **independently within each episode** — they never cross episode boundaries.
For training, extra margin is reserved for temporal jitter augmentation.

In [ ]:
train_indices = compute_window_indices(
    splits['train'], config.T_in, config.T_out, config.stride,
    jitter_margin=config.jitter_margin,
)
val_indices = compute_window_indices(
    splits['val'], config.T_in, config.T_out, config.stride,
    jitter_margin=0,
)
test_indices = compute_window_indices(
    splits['test'], config.T_in, config.T_out, config.stride,
    jitter_margin=0,
)

print('Window Index Summary')
print('=' * 50)
print(f'Train: {len(train_indices):>6} windows (jitter_margin={config.jitter_margin})')
print(f'Val:   {len(val_indices):>6} windows')
print(f'Test:  {len(test_indices):>6} windows')
print(f'Total: {len(train_indices) + len(val_indices) + len(test_indices):>6} windows')

# Show per-episode breakdown
print(f'\nPer-Episode Breakdown:')
for split_name, indices in [('train', train_indices), ('val', val_indices), ('test', test_indices)]:
    ep_counts = {}
    for wi in indices:
        ep_counts[wi.episode_id] = ep_counts.get(wi.episode_id, 0) + 1
    for ep_id, count in sorted(ep_counts.items()):
        print(f'  [{split_name:>5}] {ep_id}: {count} windows')

## 6. Create PyTorch Datasets

In [ ]:
# Prepare episode data dict for dataset (skeleton + joints only)
episode_data = {}
for ep_id, arrays in episode_data_raw.items():
    episode_data[ep_id] = {
        'skeleton': arrays['skeleton'],
        'joints': arrays['joints'],
    }

train_dataset = VAMDataset(train_indices, episode_data, norm_stats, config, split='train')
val_dataset = VAMDataset(val_indices, episode_data, norm_stats, config, split='val')
test_dataset = VAMDataset(test_indices, episode_data, norm_stats, config, split='test')

print('Dataset Summary')
print('=' * 50)
print(f'Train: {len(train_dataset):>6} samples (augmentation={config.augmentation_enabled})')
print(f'Val:   {len(val_dataset):>6} samples')
print(f'Test:  {len(test_dataset):>6} samples')

# Sanity check: get a sample
sample = train_dataset[0]
print(f'\nSample shapes:')
print(f'  input:  {sample["input"].shape}  (expected: [{config.T_in}, {config.input_dim}])')
print(f'  target: {sample["target"].shape}  (expected: [{config.T_out}, {config.joint_dim}])')
print(f'  dtype:  input={sample["input"].dtype}, target={sample["target"].dtype}')
print(f'  episode: {sample["episode_id"]}, start_frame: {sample["start_frame"]}')

assert sample['input'].shape == (config.T_in, config.input_dim), f'Unexpected input shape: {sample["input"].shape}'
assert sample['target'].shape == (config.T_out, config.joint_dim), f'Unexpected target shape: {sample["target"].shape}'
print('\nShape assertions passed!')

## 7. Create DataLoaders and Test a Batch

In [ ]:
dataloaders = create_dataloaders(config, train_dataset, val_dataset, test_dataset)

# Test each split
for split_name, dl in dataloaders.items():
    batch = next(iter(dl))
    print(f'{split_name:>5} batch:')
    print(f'  input:  {batch["input"].shape}')
    print(f'  target: {batch["target"].shape}')
    print(f'  episodes: {batch["episode_id"][:3]}...')
    print()

# Verify training batch shape
train_batch = next(iter(dataloaders['train']))
expected_input = (config.batch_size, config.T_in, config.input_dim)
expected_target = (config.batch_size, config.T_out, config.joint_dim)
assert train_batch['input'].shape == expected_input, f'Expected {expected_input}, got {train_batch["input"].shape}'
assert train_batch['target'].shape == expected_target, f'Expected {expected_target}, got {train_batch["target"].shape}'
print(f'Batch shape assertions passed!')
print(f'  input:  {train_batch["input"].shape}  = [batch, T_in, input_dim]')
print(f'  target: {train_batch["target"].shape}  = [batch, T_out, joint_dim]')

## 8. Visualize Sample Windows

In [ ]:
# Visualize a few training windows: input skeleton + joint angles -> target joint angles
n_samples = 3
fig = make_subplots(
    rows=n_samples, cols=2,
    subplot_titles=[f'Sample {i+1}: Input Joints (T_in={config.T_in})' if c == 0
                    else f'Sample {i+1}: Target Joints (T_out={config.T_out})'
                    for i in range(n_samples) for c in range(2)],
    horizontal_spacing=0.08,
    vertical_spacing=0.08,
)

joint_colors = ['#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']

# Sample from different parts of the dataset
sample_indices = np.linspace(0, len(train_dataset) - 1, n_samples, dtype=int)

for row, idx in enumerate(sample_indices):
    sample = train_dataset[int(idx)]
    input_tensor = sample['input'].numpy()    # [T_in, 54]
    target_tensor = sample['target'].numpy()   # [T_out, 6]

    # Input joint angles are the last 6 features of input
    input_joints = input_tensor[:, -6:]  # [T_in, 6]

    for j in range(6):
        show_legend = (row == 0)
        fig.add_trace(go.Scatter(
            x=list(range(config.T_in)), y=input_joints[:, j],
            name=f'j{j}', line=dict(color=joint_colors[j]),
            showlegend=show_legend,
        ), row=row + 1, col=1)

        fig.add_trace(go.Scatter(
            x=list(range(config.T_out)), y=target_tensor[:, j],
            name=f'j{j} (target)', line=dict(color=joint_colors[j], dash='dot'),
            showlegend=False,
        ), row=row + 1, col=2)

fig.update_layout(
    height=300 * n_samples, width=1200,
    title_text='Sample Training Windows (standardized values)',
)
fig.update_xaxes(title_text='Frame')
fig.update_yaxes(title_text='Standardized value')
fig.show()

In [ ]:
# Visualize skeleton keypoint trajectories for a sample window
sample = train_dataset[len(train_dataset) // 2]
input_tensor = sample['input'].numpy()  # [T_in, 54]
input_skeleton = input_tensor[:, :48]   # [T_in, 48]

fig = go.Figure()

# Plot skeleton at first and last input frames
for frame_idx, (frame_label, opacity) in enumerate([(0, 'First frame'), (config.T_in - 1, 'Last frame')]):
    sk = input_skeleton[frame_label].reshape(16, 3)
    fig.add_trace(go.Scatter3d(
        x=sk[:, 0], y=sk[:, 1], z=sk[:, 2],
        mode='markers',
        marker=dict(size=6, color='red' if frame_idx == 0 else 'blue'),
        name=f'{opacity} (standardized)',
    ))

    # Connections
    connections = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(3,8),(8,9),(9,10),(10,11),(0,12),(12,13),(0,14),(14,15)]
    for start, end in connections:
        fig.add_trace(go.Scatter3d(
            x=[sk[start, 0], sk[end, 0]],
            y=[sk[start, 1], sk[end, 1]],
            z=[sk[start, 2], sk[end, 2]],
            mode='lines',
            line=dict(color='red' if frame_idx == 0 else 'blue', width=3),
            showlegend=False,
        ))

fig.update_layout(
    title=f'Skeleton Motion in Input Window ({config.T_in} frames, standardized)',
    scene=dict(xaxis_title='X (std)', yaxis_title='Y (std)', zaxis_title='Z (std)', aspectmode='data'),
    width=800, height=600,
)
fig.show()

## 9. Validation Checks

In [ ]:
print('Running validation checks...\n')
all_passed = True

# 1. No cross-episode windows
print('1. Episode boundary check:')
for split_name, indices in [('train', train_indices), ('val', val_indices), ('test', test_indices)]:
    margin = config.jitter_margin if split_name == 'train' else 0
    for wi in indices:
        ep_frames = episode_data_raw[wi.episode_id]['skeleton'].shape[0]
        max_end = wi.start_frame + config.T_in + config.T_out + margin
        if max_end > ep_frames:
            print(f'  FAIL: {split_name} window in {wi.episode_id} exceeds episode bounds')
            all_passed = False
            break
    else:
        print(f'  {split_name}: PASS ({len(indices)} windows within episode bounds)')

# 2. No data leakage between splits
print('\n2. Data leakage check:')
train_ids = {ep.episode_id for ep in splits['train']}
val_ids = {ep.episode_id for ep in splits['val']}
test_ids = {ep.episode_id for ep in splits['test']}
if train_ids.isdisjoint(val_ids) and train_ids.isdisjoint(test_ids) and val_ids.isdisjoint(test_ids):
    print('  PASS: All splits have disjoint episode sets')
else:
    print('  FAIL: Overlapping episodes between splits!')
    all_passed = False

# 3. Round-trip normalization
print('\n3. Normalization round-trip check:')
test_joints = episode_data_raw[episodes[0].episode_id]['joints'][:10]
normed = (test_joints - norm_stats.joint_mean) / norm_stats.joint_std
recovered = normed * norm_stats.joint_std + norm_stats.joint_mean
max_err = np.abs(test_joints - recovered).max()
if max_err < 1e-5:
    print(f'  PASS: Max round-trip error = {max_err:.2e}')
else:
    print(f'  FAIL: Max round-trip error = {max_err:.2e} (threshold: 1e-5)')
    all_passed = False

# 4. Val/test determinism (no augmentation)
print('\n4. Val/test determinism check:')
s1 = val_dataset[0]
s2 = val_dataset[0]
if torch.equal(s1['input'], s2['input']) and torch.equal(s1['target'], s2['target']):
    print('  PASS: Val dataset returns identical results across calls')
else:
    print('  FAIL: Val dataset returns different results (augmentation leak?)')
    all_passed = False

# 5. No NaN in outputs
print('\n5. NaN check on sample batch:')
batch = next(iter(dataloaders['train']))
if not torch.isnan(batch['input']).any() and not torch.isnan(batch['target']).any():
    print('  PASS: No NaN values in sample batch')
else:
    print('  FAIL: NaN values detected in batch!')
    all_passed = False

print('\n' + '=' * 50)
print(f'Overall: {"ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED"}')
print('=' * 50)

## 10. Save Preprocessed Data to Disk

In [ ]:
output_dir = config.output_dir
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Save normalization stats
norm_stats.save(config.normalization_stats_path)
print(f'Saved normalization stats: {config.normalization_stats_path}')

# 2. Save episode arrays
ep_dir = output_dir / 'episodes'
ep_dir.mkdir(exist_ok=True)
for ep_id, data in episode_data.items():
    torch.save(data, ep_dir / f'{ep_id}.pt')
print(f'Saved {len(episode_data)} episode tensors to {ep_dir}/')

# 3. Save split assignments
splits_dict = {k: [ep.episode_id for ep in v] for k, v in splits.items()}
splits_path = output_dir / 'splits.json'
with open(splits_path, 'w') as f:
    json.dump(splits_dict, f, indent=2)
print(f'Saved split assignments: {splits_path}')

# 4. Save window indices
for split_name, indices in [('train', train_indices), ('val', val_indices), ('test', test_indices)]:
    path = output_dir / f'window_indices_{split_name}.pt'
    torch.save(indices, path)
print(f'Saved window indices for train/val/test')

# 5. Save pipeline config (for reloading during training/inference)
config_path = config.save()
print(f'Saved pipeline config: {config_path}')

print(f'\nAll data saved to {output_dir}/')
print(f'\nTo reload this config later:')
print(f'  config = DataPipelineConfig.load("{config_path}")')

## 11. Summary

In [ ]:
print('=' * 70)
print('DATA PREPARATION COMPLETE')
print('=' * 70)

print(f'\nEpisodes: {len(episodes)} total')
print(f'  Train: {len(splits["train"])} episodes, {len(train_indices)} windows')
print(f'  Val:   {len(splits["val"])} episodes, {len(val_indices)} windows')
print(f'  Test:  {len(splits["test"])} episodes, {len(test_indices)} windows')

print(f'\nTensor shapes:')
print(f'  Input:  [{config.T_in}, {config.input_dim}]  (skeleton + joints per frame)')
print(f'  Target: [{config.T_out}, {config.joint_dim}]  (future joint angles)')
print(f'  Batch:  [{config.batch_size}, ...]')

print(f'\nNormalization: Standardization only (no hip-centering)')
print(f'  Skeleton in robot_base_link frame (preserves spatial position)')
print(f'  Stats computed from training episodes only')

print(f'\nAugmentation (training only):')
print(f'  Temporal jitter: +/- {config.temporal_jitter_max} frames')
print(f'  Skeleton noise:  sigma={config.skeleton_noise_std}m')

print(f'\nSaved to: {config.output_dir}/')

print(f'\n{"=" * 70}')
print('NEXT: Proceed to 03_train_vam.ipynb')
print('=' * 70)